In [1]:
import requests
from bs4 import BeautifulSoup
import re

def clean_text(text):
    """기사 본문에서 불필요한 공백, 특수문자, 기자 이메일 등을 제거합니다."""
    if not text:
        return ""
    # 기자 이메일 및 관련 문구 제거 (패턴 예시)
    text = re.sub(r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+', '', text)
    # 연속된 공백 및 줄바꿈 정리
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def scrape_sbs_news(rss_url):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    try:
        # 1. XML 수집
        response = requests.get(rss_url, headers=headers)
        soup = BeautifulSoup(response.content, 'xml') # XML 전용 파서 사용
        items = soup.find_all('item')

        news_list = []

        for item in items:
            title = item.title.get_text()
            link = item.link.get_text()
            pub_date = item.pubDate.get_text()

            # 2. 개별 기사 페이지 접속 및 본문 클렌징
            article_res = requests.get(link, headers=headers)
            article_soup = BeautifulSoup(article_res.text, 'html.parser')
            
            # SBS 뉴스 본문 영역 선택 (클래스명은 사이트 구조에 따라 변경될 수 있음)
            content_area = article_soup.select_one('.main_text') or article_soup.select_one('.text_area')
            
            if content_area:
                # 불필요한 태그(광고, 스크립트 등) 미리 제거
                for s in content_area(['script', 'style', 'iframe', 'button']):
                    s.decompose()
                
                raw_content = content_area.get_text()
                cleaned_content = clean_text(raw_content)
            else:
                cleaned_content = "본문을 찾을 수 없습니다."

            news_list.append({
                "title": title,
                "date": pub_date,
                "content": cleaned_content,
                "link": link
            })
            
            print(f"수집 완료: {title[:20]}...")

        return news_list

    except Exception as e:
        print(f"에러 발생: {e}")
        return []

# 실행
rss_target = 'https://news.sbs.co.kr/news/SectionRssFeed.do?sectionId=07'
results = scrape_sbs_news(rss_target)

# 결과 확인 (첫 번째 기사)
if results:
    print("\n--- 클렌징 결과 샘플 ---")
    print(f"제목: {results[0]['title']}")
    print(f"본문: {results[0]['content'][:200]}...")

수집 완료: 미국, 유럽·중동에 군용기 150대 ...
수집 완료: 미국, 서안지구서 첫 영사 서비스…이...
수집 완료: 이란, 미국과 핵협상 앞두고 "타결 ...
수집 완료: 미국 법원, "오픈AI가 영업비밀 빼...
수집 완료: 중국 누리꾼, "한국은 문화 도둑국"...
수집 완료: 길거리서 무료로…"참된 스승" 쏟아진...
수집 완료: 스키 타다 '화들짝'…설원 가로질러 ...
수집 완료: 노인 도왔더니 "4,500만 원 배상...
수집 완료: 유엔총회, 우크라 지지 결의 채택…미...
수집 완료: 오늘 국정연설…"백악관, 관세 15%...
수집 완료: FBI국장이 왜 거기서 나와…미 하키...
수집 완료: "미국, 은행에 고객 시민권정보 수집...
수집 완료: 보석 도둑맞은 루브르 박물관장 끝내 ...
수집 완료: 러 매체 "한국학자 란코프 국민대 교...
수집 완료: 미국 유명앵커, 모친 실종 3주 만에...
수집 완료: 푸틴 "적들이 러시아 패배 위해 모든...
수집 완료: 브라질 남동부 폭우 강타…"사망·실종...
수집 완료: 월드컵 '한국 경기' 도시, 치안 악...
수집 완료: 백악관 당국자 "글로벌 관세, 15%...
수집 완료: 백악관 "트럼프 첫 옵션은 외교…필요...
수집 완료: 스위스, 베네수 마두로 측근 등 자산...
수집 완료: "미국의 이란 공격, 일주일 넘게 지...
수집 완료: "다카이치 측, 총선 당선자들에 수십...
수집 완료: 이란 "학생들 시위 권리 있지만 레드...
수집 완료: 독일 총리, 미국에 '무역 바주카포'...
수집 완료: '트럼프 평화위', 가자지구에 스테이...
수집 완료: "이란, 중국 초음속 대함미사일 구매...
수집 완료: 이란 혁명수비대, 미국 군사 압박에 ...
수집 완료: 러시아 "우크라 작전 4주년…목표달성...

--- 클렌징 결과 샘플 ---
제목: 미국, 유럽·중동에 군용기 150대 이동…이라크전 이후 최대
본문: ▲ 제럴드 R. 포드함 미국이 유럽과 중동 기지로 150대 넘는 군용기를